## *MMR

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Langchain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to LLM based applications."),
    Document(page_content="Chroma is used to store and serch documents embeddings."),
    Document(page_content="Embeddings are vector representation of text."),
    Document(page_content="MMR helps you to get the diverse result when doing similarity search."),
    Document(page_content="Langchain supports Chroma, FAISS, Pineccone and many more."),
]

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

## Initializing the Ollama Embedings
embeddings = OllamaEmbeddings(model='qwen3-embedding:0.6b')

## Intializing Vectore Store: FAISS

vectore_store = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [ ]:
## Enable MMR in retriever

retriever_mmr = vectore_store.as_retriever(
    search_type='mmr', # <--- This will enables MMR
    search_kwargs={"k":3, "lambda_mult":0.5} # k = top results, lambda_mult = relevance diversity balance
)

In [ ]:
query = "What is LangChain?"

result = retriever_mmr.invoke(query)

## Loop to check the result
for i in range(len(result)):
    print(f"Result {i+1}:")
    print('-'*25)
    print(result[i].page_content)
    print('-'*25,sep='\n')

Result 1:
-------------------------
LangChain is used to LLM based applications.
-------------------------
Result 2:
-------------------------
MMR helps you to get the diverse result when doing similarity search.
-------------------------
Result 3:
-------------------------
Embeddings are vector representation of text.
-------------------------


## *MQR (Multi Query Retriever)

In [42]:
from langchain_ollama import ChatOllama
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [43]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [45]:
vectore_store_mqr = FAISS.from_documents(
    documents=all_docs,
    embedding=embeddings
)

In [46]:
retriever_mqr = vectore_store_mqr.as_retriever(
    search_type='similarity', # <--- This will enables MMR
    search_kwargs={"k":5} # k = top results
)

In [47]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectore_store_mqr.as_retriever(search_kwargs={"k":5}),
    llm=ChatOllama(model='gpt-oss:120b-cloud')
)

In [48]:
# Query
query = "How to improve energy levels and maintain balance?"

In [51]:
# Retrieve results
similarity_results = retriever_mqr.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [52]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 2 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 5 ---
Deep sleep is crucial for cellular repair and emotional regulation.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve men

## *Contexual Compression Retriever

In [53]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [54]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [55]:
vec_store = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [56]:
base_retriever = vec_store.as_retriever(search_kwargs={'k':5})

In [57]:
# Creating Compressor
compressor = LLMChainExtractor.from_llm(llm=ChatOllama(model='gpt-oss:120b-cloud'))

# Creating the Contexual Compressor Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [58]:
query = 'What is Photosynthesis?'

compressed_result = compression_retriever.invoke(query)

for i, doc in enumerate(compressed_result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
Photosynthesis does not occur in animal cells.

--- Result 3 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
